In [1]:
import re

# --- PART 1: GENETIC MAPPING ---
GENETIC_CODE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M', 'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K', 'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L', 'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A', 'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G', 'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L', 'TAC':'Y', 'TAT':'Y', 'TAA':'_', 'TAG':'_',
    'TGA':'_', 'GTC':'V', 'GTT':'V', 'GTA':'V', 'GTG':'V', 'TGC':'C', 'TGT':'C', 'TGG':'W',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R', 'CAT':'H', 'CAC':'H', 'CAA':'Q', 'CAG':'Q'
}

# --- PART 2: PROCESSING ENGINE ---

def clean_sequence(raw_data: str) -> str:
    """Filtra ruido genómico y estandariza a mayúsculas usando expresiones regulares."""
    clean_data = raw_data.upper()
    return re.sub(r'[^ATGC]', '', clean_data)

def translate_dna(dna_seq: str, stop_at_stop_codon: bool = False) -> str:
    """Traduce tripletes de ADN a la estructura primaria de la proteína."""
    protein = []
    for i in range(0, (len(dna_seq) // 3) * 3, 3):
        codon = dna_seq[i:i+3]
        aa = GENETIC_CODE.get(codon, "?")
        if stop_at_stop_codon and aa == '_':
            break
        protein.append(aa)
    return "".join(protein)

def get_reverse_complement(dna_seq: str) -> str:
    """Genera la cadena reversa complementaria de ADN."""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return "".join(complement.get(base, base) for base in reversed(dna_seq))

def translate_6_frames(dna_seq: str) -> dict:
    """Traduce la secuencia de ADN en los 6 marcos de lectura posibles (+1, +2, +3, -1, -2, -3)."""
    rev_dna = get_reverse_complement(dna_seq)
    frames = {}

    # Marcos directos (+1, +2, +3)
    frames['+1'] = translate_dna(dna_seq[0:])
    frames['+2'] = translate_dna(dna_seq[1:])
    frames['+3'] = translate_dna(dna_seq[2:])

    # Marcos reversos (-1, -2, -3)
    frames['-1'] = translate_dna(rev_dna[0:])
    frames['-2'] = translate_dna(rev_dna[1:])
    frames['-3'] = translate_dna(rev_dna[2:])

    return frames

# --- PART 3: ADVANCED MOLECULAR DIAGNOSTIC ENGINE (HBB) ---

def run_hbb_diagnostic(protein_seq: str) -> str:
    """
    Detector de biomarcadores moleculares para el gen HBB.
    Analiza la Posición 6 biológica ajustando según la Metionina iniciadora.
    """
    if not protein_seq:
        return "ERROR: Secuencia vacía."

    # Ajuste de índice: Posición 6 biológica es índice 6 si empieza con Metionina (M)
    target_idx = 6 if protein_seq.startswith('M') else 5

    if len(protein_seq) > target_idx:
        target_aa = protein_seq[target_idx]
        pos_num = target_idx + 1

        report = ["\n" + "*"*50, " [REPORTE DE DIAGNÓSTICO MOLECULAR: GEN HBB] ", "*"*50]

        if target_aa == 'E':
            report.append("DIAGNÓSTICO: SIN MUTACIÓN (Fenotipo Normal - HbA)")
            report.append(f"PROTEÍNA: Hemoglobina Beta Salvaje (Wild Type)")
            report.append(f"DETALLE MOLECULAR: Ácido Glutámico ('E') en Posición {pos_num}.")

        elif target_aa == 'V':
            report.append("DIAGNÓSTICO: ANEMIA FALCIFORME (Sickle Cell Anemia - HbS)")
            report.append("MUTACIÓN DETECTADA: Cambio Glu6Val (Sustitución por Valina)")
            report.append(f"DETALLE MOLECULAR: Valina ('V') patogénica en Posición {pos_num}.")

        elif target_aa == 'K':
            report.append("DIAGNÓSTICO: HEMOGLOBINOPATÍA C (Enfermedad por HbC)")
            report.append("MUTACIÓN DETECTADA: Cambio Glu6Lys (Sustitución por Lisina)")
            report.append(f"DETALLE MOLECULAR: Lisina ('K') patogénica en Posición {pos_num}.")

        else:
            report.append(f"DIAGNÓSTICO: VARIANTE ATÍPICA ('{target_aa}')")
            report.append(f"HALLAZGO: Aminoácido no estándar detectado en Posición {pos_num}.")
            report.append("ADVERTENCIA: Se requiere revisión clínica.")

        report.append("*"*50)
        return "\n".join(report)
    else:
        return "\nERROR CRÍTICO: La secuencia es demasiado corta para analizar el gen HBB."

In [2]:
!pip install gradio -q
import gradio as gr

def analizar_adn_gui(secuencia_adn):
    clean_dna = clean_sequence(secuencia_adn)
    if not clean_dna:
        return "Error: No se detectaron bases válidas (A, T, G, C).", "", ""

    # 1. Traducir en 6 Marcos
    frames = translate_6_frames(clean_dna)

    texto_marcos = (
        f"--- CADENA DIRECTA ---\n"
        f"Marco +1: {frames['+1']}\n"
        f"Marco +2: {frames['+2']}\n"
        f"Marco +3: {frames['+3']}\n\n"
        f"--- CADENA REVERSA COMPLEMENTARIA ---\n"
        f"Marco -1: {frames['-1']}\n"
        f"Marco -2: {frames['-2']}\n"
        f"Marco -3: {frames['-3']}"
    )

    # 2. Métricas y Diagnóstico en Marco +1
    gc_val = (clean_dna.count("G") + clean_dna.count("C")) / len(clean_dna) * 100
    metrics = f"Longitud: {len(clean_dna)} bp | Contenido GC: {gc_val:.2f}%\n"
    diag = run_hbb_diagnostic(frames['+1'])

    return clean_dna, texto_marcos, f"{metrics}\n{diag}"

demo = gr.Interface(
    fn=analizar_adn_gui,
    inputs=gr.Textbox(lines=4, placeholder="Pega tu secuencia de ADN aquí...", label="Entrada ADN"),
    outputs=[
        gr.Textbox(label="Secuencia Limpia"),
        gr.Textbox(label="Traducción en 6 Marcos de Lectura (6 Frames)", lines=8),
        gr.Textbox(label="Métricas y Diagnóstico Molecular (Marco +1)", lines=8)
    ],
    title="Biotech Sequence Suite - 6 Frames & Diagnostic",
    description="Analizador genómico avanzado con traducción en 6 marcos de lectura y diagnóstico molecular de variantes HbA, HbS y HbC."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b16bb0935032ea3b51.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
